In [5]:
from langchain_core.documents import Document
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_community.vectorstores import FAISS

from llm_llamacpp import embedder, llm

In [2]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [4]:
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedder
)

In [6]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=vectorstore.as_retriever(search_kwargs={'k':5}),
    base_compressor=LLMChainExtractor.from_llm(llm)
)

In [11]:
query = 'Tell me about basketball'

results = compression_retriever.invoke(query)

In [12]:
print('According to Contextual Compressor, the retrieved documents are:')
for i, doc in enumerate(results):
    print(f'Document no {i+1}\n{doc}', end='\n\n')

According to Contextual Compressor, the retrieved documents are:
Document no 1
page_content='Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league.' metadata={'source': 'Doc3'}

Document no 2
page_content='No part of the context is relevant to the question. The provided context discusses medieval castles, plant biology (chlorophyll), and siege weapons, none of which address basketball. Therefore, per the instruction to return NO_OUTPUT when none of the context is relevant:

NO_OUTPUT' metadata={'source': 'Doc2'}



#### HAHAHAHA This is so damn fragile lmao